# BIO 2910 · Assignment 07-02 · MSA Homework
## Opsins, optogenetics and the 2026 Nobel Prize

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JakeSaunders/BIO2910-Bioinformatics/blob/main/Unit07/BIO2910_Assignment_07-02_MSA_homework.ipynb)

**First: File → Save a copy in Drive.** Then run the cells in order (Shift + Enter).

---

### Why opsins, why now?
The **2026 Nobel Prize in Physiology or Medicine** went to **Karl Deisseroth, Peter Hegemann and Georg Nagel** for discovering light-gated ion channels and developing **optogenetics**.

The story starts with a green alga, *Chlamydomonas reinhardtii*, which swims toward light. Hegemann and Nagel showed that its eyespot uses **channelrhodopsins**: proteins that are a light sensor *and* an ion channel in one molecule. Shine blue light on **channelrhodopsin-2 (ChR2)** and it opens to let cations in. In 2005 Deisseroth's lab put ChR2 into mammalian neurons and showed that a flash of blue light could make them fire action potentials with millisecond precision. Today neuroscientists put channelrhodopsins in specific neuron types to turn them **on**, and use light-driven pumps like **halorhodopsin** to turn them **off**.

### Two families that "look" the same
Every protein in this homework is a **rhodopsin**: a protein with **7 transmembrane (7TM) helices** that holds the light-absorbing molecule **retinal**, attached to a **lysine** in helix 7.

| | **Animal opsins** (type II) | **Microbial rhodopsins** (type I) |
|---|---|---|
| examples here | human rod rhodopsin, the 3 cone opsins, melanopsin, cow rhodopsin, fruit fly Rh1, octopus rhodopsin | **bacteriorhodopsin** (archaea, H⁺ pump), **halorhodopsin** (archaea, Cl⁻ pump), **channelrhodopsin-2** (alga, cation channel) |
| what they do | G protein–coupled receptors: light → signaling cascade → vision | light **directly** moves ions across the membrane |
| famous for | how you see | bacteriorhodopsin (discovered in the early 1970s) was the first; channelrhodopsins → optogenetics |

**The big question for this homework:** both families have 7 TMs and a retinal-binding lysine in TM7. Are they **related** (homologous, from one ancestral protein) or did evolution **build the same solution twice** (convergence)? Multiple sequence alignment is one of our best tools for asking that.

### What you'll do
1. Make a **hydropathy plot** for one protein and check it against the TM helices annotated in **UniProt**, using Python to pull the annotations straight from UniProt's website
2. Repeat for all 11 proteins
3. Align the **animal opsins** and find the conserved retinal-binding lysine
4. Align **everything**, then test whether animal and microbial rhodopsins share detectable sequence similarity, using a **shuffled-sequence control**

---
# Setup

In [ ]:
!pip install -q pyfamsa pymsaviz

In [ ]:
import urllib.request, random, requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyfamsa import Aligner, Sequence
from pymsaviz import MsaViz

BASE = "https://raw.githubusercontent.com/JakeSaunders/BIO2910-Bioinformatics/main/data/"


def read_fasta(url):
    """Read a FASTA file from a URL. Return a dictionary of {name: sequence}."""
    text = urllib.request.urlopen(url).read().decode()
    seqs = {}
    for line in text.splitlines():
        line = line.strip()
        if line.startswith(">"):
            name = line[1:].split()[0]
            seqs[name] = ""
        elif line:
            seqs[name] += line.upper()
    return seqs


def align(seqs):
    """Align a {name: sequence} dictionary with FAMSA. Return {name: aligned sequence}."""
    msa = Aligner(guide_tree="upgma").align([Sequence(n.encode(), s.encode()) for n, s in seqs.items()])
    aligned = {m.id.decode(): m.sequence.decode() for m in msa}
    return {n: aligned[n] for n in seqs}


def write_fasta(seqs, filename):
    with open(filename, "w") as f:
        for name, s in seqs.items():
            f.write(f">{name}\n{s}\n")


def percent_identity(a, b):
    """Percent identity of two ALIGNED sequences, ignoring columns with a gap in either."""
    compared = same = 0
    for x, y in zip(a, b):
        if x != "-" and y != "-":
            compared += 1
            same += (x == y)
    return 100 * same / compared

print("Ready!")

In [ ]:
opsins = read_fasta(BASE + "opsins.fasta")

# The headers also hold each protein's UniProt accession number. Let's pull those out.
header_text = urllib.request.urlopen(BASE + "opsins.fasta").read().decode()
accession = {}
for line in header_text.splitlines():
    if line.startswith(">"):
        name = line[1:].split()[0]
        accession[name] = line.split("UniProt:")[1].split()[0]
        print(line)

animal = ["Rhodopsin_human", "Rhodopsin_cow", "BlueOpsin_human", "GreenOpsin_human",
          "RedOpsin_human", "Melanopsin_human", "Rh1_fruitfly", "Rhodopsin_octopus"]
microbial = ["Bacteriorhodopsin", "Halorhodopsin", "Channelrhodopsin2"]

---
# Part 1 · Hydropathy of a single protein

A TM helix needs ~20 hydrophobic amino acids in a row to span the lipid bilayer. The **Kyte–Doolittle** scale gives every amino acid a score (positive = hydrophobic). If you average the scores in a **sliding window** of 19 residues, TM helices show up as peaks. This is the same calculation you did in R last year; only the language has changed.

In [ ]:
KD = {"A":  1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C":  2.5,
      "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I":  4.5,
      "L":  3.8, "K": -3.9, "M":  1.9, "F":  2.8, "P": -1.6,
      "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V":  4.2}


def hydropathy(seq, window=19):
    """Kyte-Doolittle sliding-window average. Returns one value per residue (NaN at the ends)."""
    scores = [KD[aa] for aa in seq]
    half = window // 2
    smooth = [float("nan")] * len(seq)
    for i in range(half, len(seq) - half):
        smooth[i] = sum(scores[i - half : i + half + 1]) / window
    return smooth


rho = opsins["Rhodopsin_human"]
rho_kd = hydropathy(rho)
print(len(rho), "residues;  first few smoothed values:", [round(x, 2) for x in rho_kd[9:14]])

### 1.1 Ask UniProt where the helices are
**UniProt** (you met it in the Databases unit) records experimentally determined or predicted features of each protein, including **Transmembrane** regions. Websites like UniProt have an **API**: a URL that returns data a program can read instead of a web page. The `requests` library fetches it, and `.json()` turns the reply into Python lists and dictionaries.

In [ ]:
def uniprot_tm(acc):
    """Return a list of (start, end) transmembrane regions for a UniProt accession."""
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.json"
    data = requests.get(url, timeout=30).json()
    tms = []
    for feature in data["features"]:
        if feature["type"] == "Transmembrane":
            tms.append((feature["location"]["start"]["value"], feature["location"]["end"]["value"]))
    return tms


TM = {}
try:
    for name, acc in accession.items():
        TM[name] = uniprot_tm(acc)
    print("Downloaded TM annotations from UniProt")
except Exception as err:          # if UniProt is down, use the copy saved in the course repo
    print("UniProt unavailable, using the saved copy:", err)
    tm_table = pd.read_csv(BASE + "opsins_TM_uniprot.tsv", sep="\t")
    for name in opsins:
        rows = tm_table[tm_table["name"] == name]
        TM[name] = list(zip(rows["start"], rows["end"]))

for name, regions in TM.items():
    print(f"{name:18s} {len(regions)} TM regions: {regions}")

In [ ]:
def plot_hydropathy(name, ax=None, window=19):
    """Plot one protein's hydropathy profile with its UniProt TM helices shaded."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 3.5))
    kd = hydropathy(opsins[name], window)
    pos = range(1, len(kd) + 1)
    ax.plot(pos, kd, color="black", linewidth=1)
    ax.fill_between(pos, kd, 0, where=[k > 0 for k in kd], color="#c0392b", alpha=0.5)
    ax.fill_between(pos, kd, 0, where=[k < 0 for k in kd], color="#2980b9", alpha=0.4)
    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
    ax.axhline(1.6, color="#c0392b", linestyle=":", linewidth=0.8)   # rough TM threshold
    for start, end in TM[name]:
        ax.axvspan(start, end, color="gold", alpha=0.35)
    ax.set_title(f"{name}  ({len(TM[name])} TM helices in UniProt)", fontsize=10)
    ax.set_xlabel("Residue")
    ax.set_ylabel("KD score")
    return ax


plot_hydropathy("Rhodopsin_human")
plt.tight_layout()
plt.savefig("rhodopsin_hydropathy.png", dpi=150)
plt.show()

> **✏️ Question 1:** Red = hydrophobic, blue = hydrophilic, gold bars = UniProt's TM helices, dotted line = a common TM threshold (1.6). How many hydrophobic peaks rise above the dotted line? Do they line up with UniProt's 7 TM helices? Which helix is **hardest** to see from hydropathy alone?

---
# Part 2 · All 11 rhodopsins

In [ ]:
fig, axes = plt.subplots(len(opsins), 1, figsize=(12, 2.2 * len(opsins)), sharex=True)
for ax, name in zip(axes, animal + microbial):
    plot_hydropathy(name, ax=ax)
    ax.set_xlabel("")
axes[-1].set_xlabel("Residue")
plt.tight_layout()
plt.savefig("all_opsins_hydropathy.png", dpi=120)
plt.show()

> **✏️ Question 2:** Do the **microbial** rhodopsins (bottom three) show the same 7-peak pattern as the animal opsins? Count the peaks in bacteriorhodopsin.
>
> **✏️ Question 3:** UniProt lists 7 TM helices for every protein **except channelrhodopsin-2**, where it lists only 3. Look at the ChR2 plot: does the hydropathy suggest only 3 helices? Now look up ChR2 (Q8RUT8) on uniprot.org. Is the entry **reviewed** (Swiss-Prot) or **unreviewed** (TrEMBL), and what program made its TM predictions? What does this tell you about trusting database annotations?
>
> **✏️ Question 4:** Octopus rhodopsin and melanopsin have long stretches **after** TM7 that are strongly hydrophilic. What part of the protein is that, and which side of the membrane is it on?

---
# Part 3 · Align the animal opsins

Now we'll align the 8 animal opsins and average the hydropathy **down each column** of the alignment. Then we'll find the **lysine that binds retinal**. In human rhodopsin it is **Lys296**, in TM7.

In [ ]:
animal_aln = align({n: opsins[n] for n in animal})
write_fasta(animal_aln, "animal_opsins_alignment.fasta")
aln_len = len(animal_aln["Rhodopsin_human"])
print("Alignment length:", aln_len)

# Mean KD score in each column (ignoring gaps), then a sliding window
col_means = []
for col in range(aln_len):
    residues = [s[col] for s in animal_aln.values() if s[col] != "-"]
    col_means.append(sum(KD[aa] for aa in residues) / len(residues))

profile = pd.Series(col_means).rolling(19, center=True).mean()

In [ ]:
def aln_column(aligned_seq, residue_number):
    """Find which alignment column holds residue number N (1-based) of an aligned sequence."""
    count = 0
    for col, aa in enumerate(aligned_seq):
        if aa != "-":
            count += 1
            if count == residue_number:
                return col
    return None


k_col = aln_column(animal_aln["Rhodopsin_human"], 296)
print("Rhodopsin Lys296 is in alignment column", k_col + 1)   # +1 to count from 1 like a human
print()
for name, s in animal_aln.items():
    print(f"{name:18s} residue in that column: {s[k_col]}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(range(1, aln_len + 1), profile, color="black", linewidth=1)
ax.fill_between(range(1, aln_len + 1), profile, 0, where=profile > 0, color="#c0392b", alpha=0.5)
ax.fill_between(range(1, aln_len + 1), profile, 0, where=profile < 0, color="#2980b9", alpha=0.4)
ax.axhline(0, color="grey", linestyle="--", linewidth=0.8)
ax.axhline(1.6, color="#c0392b", linestyle=":", linewidth=0.8)
ax.axvline(k_col + 1, color="green", linewidth=2, label="retinal-binding Lys (Rho K296)")
ax.set_xlabel("Alignment column")
ax.set_ylabel("Mean KD score")
ax.set_title("Animal opsins: mean hydropathy across the alignment")
ax.legend()
plt.tight_layout()
plt.savefig("animal_opsins_profile.png", dpi=150)
plt.show()

In [ ]:
# Zoom in on TM7 and the lysine
mv = MsaViz("animal_opsins_alignment.fasta", color_scheme="Hydrophobicity",
            start=k_col + 1 - 30, end=k_col + 1 + 30, show_consensus=True)
mv.add_markers([k_col + 1], color="green", marker="v")
fig = mv.plotfig()
fig.savefig("animal_opsins_TM7.png", dpi=150, bbox_inches="tight")

> **✏️ Question 5:** Is the retinal-binding lysine conserved in all 8 animal opsins? Why would natural selection keep this one residue unchanged in fruit flies, octopuses and humans?
>
> **✏️ Question 6:** Compare the 7 peaks in the aligned profile with the single-protein plot from Part 1. Is the 7TM pattern **clearer** in the alignment than it was with the mGluRs last year? Why do shorter proteins with fewer extra domains give a cleaner result?

---
# Part 4 · Are animal and microbial rhodopsins related?

Let's throw all 11 proteins into one alignment and measure percent identity between every pair.

**But what does a percent identity mean?** Even two **random** protein sequences match at some positions by chance, and an aligner will happily insert gaps to make them match more. To know what "no relationship" looks like, we need a **control**: we'll shuffle the order of the amino acids in bacteriorhodopsin. That keeps its composition (same number of each amino acid) but destroys any real evolutionary signal. Then we align it against three proteins and compare the real % identity to the shuffled ones:
* **human rhodopsin**: the question we actually care about
* **halorhodopsin**: a microbial rhodopsin, so a **positive control**
* **channelrhodopsin-2**: another microbial rhodopsin

In [ ]:
all_aln = align(opsins)
write_fasta(all_aln, "all_opsins_alignment.fasta")
names = animal + microbial

pid = pd.DataFrame([[percent_identity(all_aln[a], all_aln[b]) for b in names] for a in names],
                   index=names, columns=names)

plt.figure(figsize=(9, 7.5))
sns.heatmap(pid, annot=True, fmt=".0f", cmap="viridis", vmin=0, vmax=100,
            cbar_kws={"label": "% identity"})
plt.title("Percent identity: animal opsins (top-left) vs. microbial rhodopsins (bottom-right)")
plt.tight_layout()
plt.savefig("opsin_identity_heatmap.png", dpi=150)
plt.show()

In [ ]:
random.seed(2910)          # makes the "random" shuffles the same for everyone
br = opsins["Bacteriorhodopsin"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.2), sharey=True)
for ax, other in zip(axes, ["Rhodopsin_human", "Halorhodopsin", "Channelrhodopsin2"]):
    # the real pair
    real = percent_identity(*align({other: opsins[other], "BR": br}).values())
    # 20 shuffled versions of bacteriorhodopsin
    shuffled_scores = []
    for i in range(20):
        shuffled = "".join(random.sample(br, len(br)))       # same amino acids, random order
        pair = align({other: opsins[other], "Shuffled_BR": shuffled})
        shuffled_scores.append(percent_identity(*pair.values()))
    print(f"BR vs {other:18s} real: {real:5.1f}%   shuffled: {min(shuffled_scores):.1f}-{max(shuffled_scores):.1f}%")

    ax.hist(shuffled_scores, bins=8, color="grey", label="shuffled BR")
    ax.axvline(real, color="red", linewidth=2, label="real BR")
    ax.set_title(f"Bacteriorhodopsin vs. {other}", fontsize=10)
    ax.set_xlabel("% identity")
    ax.set_xlim(10, 40)
axes[0].set_ylabel("Count")
axes[0].legend()
plt.tight_layout()
plt.savefig("shuffle_control.png", dpi=150)
plt.show()

> **✏️ Question 7:** In the heatmap, what range of % identity do you see **within** the animal opsins? **Within** the microbial rhodopsins? **Between** the two groups?
>
> **✏️ Question 8:** For each of the three panels, is the real % identity clearly **above** the shuffled range, or **inside** it? Which pair has a sequence relationship you can detect, and which pairs look no different from random?
>
> **✏️ Question 9:** Channelrhodopsin-2 is a microbial rhodopsin. Its 3D structure (solved by X-ray crystallography) folds almost exactly like bacteriorhodopsin's, so biologists are confident the two are related. Yet in your test, its sequence identity to BR is no better than random. What does this tell you about the limits of sequence alignment? Does "no detectable similarity" prove "not related"?
>
> **✏️ Question 10:** Microbial and animal rhodopsins share 7 TMs and a retinal-binding lysine in TM7, but (as you just showed) no detectable sequence similarity. Give one argument for **common ancestry** and one for **convergent evolution**. (Biologists still debate this, so there's no single right answer. Your reasoning is what counts.)
>
> **✏️ Question 11:** Channelrhodopsin-2 came from an alga, not an animal. Why was it useful that optogenetics borrowed a **microbial** rhodopsin rather than an animal opsin to make neurons fire? (Hint: compare *what each family does* in the table at the top.)

---
# Part 5 · Window size

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, w in zip(axes, [5, 11, 19]):
    plot_hydropathy("Bacteriorhodopsin", ax=ax, window=w)
    ax.set_title(f"Bacteriorhodopsin, window = {w}", fontsize=10)
plt.tight_layout()
plt.savefig("window_size.png", dpi=150)
plt.show()

> **✏️ Question 12:** What happens to the hydropathy plot as the window gets smaller? With which window are the 7 TM helices easiest to count? Why does a window of about 19 work well for TM helices? (Hint: how many residues does it take to cross a membrane as an α-helix?)

---
### What to turn in
Answer **Questions 1–12** in a Word document and include these figures (📁 Files panel on the left → right-click → Download):
* `rhodopsin_hydropathy.png`
* `animal_opsins_TM7.png`
* `opsin_identity_heatmap.png`
* `shuffle_control.png`